In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
from dataloader import Loader , SpecZNorm , STFTSpec
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.classification import (
    BinaryAccuracy, BinaryRecall, BinaryPrecision, BinaryF1Score, BinaryAUROC
)

c:\Users\kakas\miniconda3\envs\EEG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
transform = nn.Sequential(
    STFTSpec(fs=250, n_fft=256, hop_length=64, fmin=1.0, fmax=40.0),
    SpecZNorm(),
)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dl=Loader(transform=transform ,batch_size=128)
dl=dl.return_Loader()

In [4]:
print('before ')
batch = next(iter(dl))
print('after')
x = batch["x"].float().to(device)  # [B, C, F, TT]
y = batch["y"].float().to(device)
if y.ndim == 1:
    y = y.unsqueeze(1)

print("x:", x.shape, "y:", y.shape)

before 
after
x: torch.Size([128, 41, 39, 40]) y: torch.Size([128, 1])


In [5]:
in_ch= x.shape[1]
m = models.resnet18()  
# Replace stem: smaller kernel, no aggressive downsampling
m.conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, stride=1, padding=1, bias=False)
m.maxpool = nn.Identity()  # remove maxpool

# Binary head
m.fc = nn.Linear(m.fc.in_features, 1)

m = m.to(device) 
with torch.no_grad():
    logits = m(x)
print("logits:", logits.shape)  # should be [B, 1]

logits: torch.Size([128, 1])


In [8]:
criterion = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-2)
state_dict = torch.load('model_weights.pth', weights_only=True)  
m.load_state_dict(state_dict)  # Load the weights into the model

<All keys matched successfully>

In [9]:
m.train()
batch = next(iter(dl))
x = batch["x"].float().to(device)
y = batch["y"].float().to(device)
if y.ndim == 1:
    y = y.unsqueeze(1)

logits = m(x)
loss = criterion(logits, y)

opt.zero_grad(set_to_none=True)
loss.backward()
opt.step()

print("loss:", float(loss))

loss: 0.7206263542175293


In [10]:
writer = SummaryWriter("runs/resnet2d_spec")
global_step = 0

In [11]:
def ensure_2d_y(y):
    # make y shape [B,1] float
    if y.ndim == 1:
        y = y.unsqueeze(1)
    return y.float()


In [12]:
def ensure_2d_y(y):
    # y -> [B,1] float
    if y.ndim == 1:
        y = y.unsqueeze(1)
    return y.float()

def train_only(model, train_dl, epochs=10, lr=3e-4, weight_decay=1e-2,
               pos_weight=None, threshold=0.5, log_every=50):

    model = model.to(device)

    # loss
    if pos_weight is not None:
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    else:
        criterion = nn.BCEWithLogitsLoss()

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train()

        # epoch metrics
        acc = BinaryAccuracy(threshold=threshold).to(device)
        rec = BinaryRecall(threshold=threshold).to(device)
        pre = BinaryPrecision(threshold=threshold).to(device)
        f1  = BinaryF1Score(threshold=threshold).to(device)
        auc = BinaryAUROC().to(device)

        total_loss = 0.0
        total_n = 0

        for i, batch in enumerate(train_dl, start=1):
            x = batch["x"].to(device).float()          # [B,C,F,TT]
            y = ensure_2d_y(batch["y"].to(device))     # [B,1]

            logits = model(x)                          # [B,1]
            loss = criterion(logits, y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            probs = torch.sigmoid(logits)

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_n += bs

            # update metrics
            y_int = y.int()
            acc.update(probs, y_int)
            rec.update(probs, y_int)
            pre.update(probs, y_int)
            f1.update(probs, y_int)
            auc.update(probs, y_int)

            # step logging (optional)
            if (i % log_every) == 0:
                writer.add_scalar("train/loss_step", loss.item(), global_step)
            global_step += 1

        # epoch stats
        epoch_loss = total_loss / max(total_n, 1)
        epoch_acc = acc.compute().item()
        epoch_rec = rec.compute().item()
        epoch_pre = pre.compute().item()
        epoch_f1  = f1.compute().item()
        epoch_auc = auc.compute().item()

        # epoch logging
        writer.add_scalar("train/loss", epoch_loss, epoch)
        writer.add_scalar("train/acc", epoch_acc, epoch)
        writer.add_scalar("train/recall", epoch_rec, epoch)
        writer.add_scalar("train/precision", epoch_pre, epoch)
        writer.add_scalar("train/f1", epoch_f1, epoch)
        writer.add_scalar("train/auc", epoch_auc, epoch)

        print(
            f"Epoch {epoch:02d} | "
            f"loss {epoch_loss:.4f} acc {epoch_acc:.3f} "
            f"rec {epoch_rec:.3f} pre {epoch_pre:.3f} f1 {epoch_f1:.3f} auc {epoch_auc:.3f}"
        )

    writer.close()


In [14]:
train_only(m, dl, epochs=10)

Epoch 01 | loss 0.3689 acc 0.843 rec 0.617 pre 0.790 f1 0.693 auc 0.884
Epoch 02 | loss 0.3560 acc 0.849 rec 0.637 pre 0.797 f1 0.708 auc 0.893
Epoch 03 | loss 0.3428 acc 0.855 rec 0.653 pre 0.803 f1 0.720 auc 0.901
Epoch 04 | loss 0.3275 acc 0.863 rec 0.676 pre 0.815 f1 0.739 auc 0.910
Epoch 05 | loss 0.3145 acc 0.870 rec 0.695 pre 0.824 f1 0.754 auc 0.917
Epoch 06 | loss 0.3020 acc 0.876 rec 0.713 pre 0.831 f1 0.768 auc 0.924
Epoch 07 | loss 0.2972 acc 0.879 rec 0.720 pre 0.835 f1 0.773 auc 0.926
Epoch 08 | loss 0.2868 acc 0.886 rec 0.735 pre 0.846 f1 0.787 auc 0.931
Epoch 09 | loss 0.2699 acc 0.892 rec 0.749 pre 0.856 f1 0.799 auc 0.939
Epoch 10 | loss 0.2596 acc 0.896 rec 0.761 pre 0.860 f1 0.808 auc 0.944


In [15]:
torch.save(m.state_dict(), 'model_weights.pth')

In [22]:
test_loader=Loader('cache_windows_eval/manifest.jsonl',transform=transform,
        batch_size=32,
        shuffle=False, # it's the issue
        num_workers=2,
        pin_memory=False,)
test_loader=test_loader.return_Loader()

In [23]:
m.eval()


ResNet(
  (conv1): Conv2d(41, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), 

In [24]:
import torch
from collections import defaultdict

all_probs = []
all_y = []
by_record = defaultdict(list)  # stem -> list of probs (or logits)

with torch.no_grad():
    for batch in test_loader:
        x = batch["x"].to(device).float()              # [B,C,F,T] or [B,C,T]
        y = batch.get("y", None)
        stem = batch.get("stem", None)                 # optional

        logits = m(x).squeeze(1)                   # [B]
        probs = torch.sigmoid(logits)                  # [B]

        all_probs.append(probs.cpu())

        if y is not None:
            all_y.append(y.cpu().float())

        if stem is not None:
            # stem might be a list of strings length B
            for s, p in zip(stem, probs.cpu().tolist()):
                by_record[s].append(p)


In [25]:
all_probs = torch.cat(all_probs).numpy()
all_y = torch.cat(all_y).numpy() if len(all_y) else None


In [26]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

auc = roc_auc_score(all_y, all_probs)

# choose a threshold (0.5 is not always best!)
thr = 0.5
pred = (all_probs >= thr).astype(int)

pre, rec, f1, _ = precision_recall_fscore_support(all_y, pred, average="binary")
print("AUC:", auc, "Precision:", pre, "Recall:", rec, "F1:", f1)


AUC: 0.7951051075984306 Precision: 0.4977394744278045 Recall: 0.5596505162827641 F1: 0.5268825244896433
